In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter
from nltk.tokenize import word_tokenize
import nltk
import numpy as np
import os
import re

nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
input_folder = './data/raw/'
output_folder = './data-overview-analysis/bar_chart_distribution/'
os.makedirs(output_folder, exist_ok=True)  
csv_files = ['train.csv', 'validation.csv', 'test.csv']

In [7]:
columns = ['Question', 'Answer']

# Bar Chart - Visualization Lengths

In [8]:
# Color mapping for columns
column_colors = {'Question': 'skyblue', 'Answer': 'lightcoral'}

# Parameter: Maximum word count to display (set to None to use data's max)
max_word_count_display = None

for csv_file in csv_files:
    # Read CSV
    file_path = os.path.join(input_folder, csv_file)
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"File {file_path} not found. Skipping...")
        continue

    dataset_name = csv_file.replace('.csv', '')

    for column_name in columns:
        if column_name not in df.columns:
            print(f"Column {column_name} not found in {csv_file}. Skipping...")
            continue

        # Count words in each row (split by whitespace)
        word_counts = df[column_name].astype(str).apply(lambda x: len(x.split()))

        # Remove zero counts
        word_counts = word_counts[word_counts > 0]
        if word_counts.empty:
            print(f"No valid data in column {column_name} of {csv_file}. Skipping...")
            continue

        # Count frequency of each word count
        count_freq = Counter(word_counts)

        # Determine min and max word counts
        min_count = min(count_freq.keys())
        max_count = max(count_freq.keys()) if max_word_count_display is None else min(max_word_count_display, max(count_freq.keys()))

        # Create x-axis with all integers from min to max
        x_values = list(range(int(min_count), int(max_count) + 1))
        y_values = [count_freq.get(x, 0) for x in x_values]

        # Create bar chart with column-specific color
        plt.figure(figsize=(10, 6))
        plt.bar(x_values, y_values, color=column_colors[column_name], edgecolor='black')
        plt.xlabel('Number of words')
        plt.ylabel('Frequency')
        plt.title(f'{column_name} Length Distribution - {dataset_name}')
        plt.xticks(x_values[::max(1, len(x_values)//20)])
        bar_output_path = os.path.join(output_folder, f'{dataset_name}_{column_name.lower()}_word_count_distribution.png')
        plt.savefig(bar_output_path, bbox_inches='tight')
        plt.close()
        print(f"Saved bar chart for {column_name} ({dataset_name}) to '{bar_output_path}'")

Saved bar chart for Question (train) to './data-overview-analysis/bar_chart_distribution/train_question_word_count_distribution.png'
Saved bar chart for Answer (train) to './data-overview-analysis/bar_chart_distribution/train_answer_word_count_distribution.png'
Saved bar chart for Question (validation) to './data-overview-analysis/bar_chart_distribution/validation_question_word_count_distribution.png'
Saved bar chart for Answer (validation) to './data-overview-analysis/bar_chart_distribution/validation_answer_word_count_distribution.png'
Saved bar chart for Question (test) to './data-overview-analysis/bar_chart_distribution/test_question_word_count_distribution.png'
Saved bar chart for Answer (test) to './data-overview-analysis/bar_chart_distribution/test_answer_word_count_distribution.png'


# World Cloud - Visualization Population

In [15]:
output_folder = './data-overview-analysis/new/world_cloud/'
os.makedirs(output_folder, exist_ok=True)

In [17]:
for csv_file in csv_files:
    # Read CSV
    file_path = os.path.join(input_folder, csv_file)
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"File {file_path} not found. Skipping...")
        continue

    dataset_name = csv_file.replace('.csv', '')

    for column_name in columns:
        if column_name not in df.columns:
            print(f"Column {column_name} not found in {csv_file}. Skipping...")
            continue

        # Collect all words, excluding "the" and the first word of each row
        all_words = []
        for text in df[column_name].astype(str):
            words = text.lower().split()  # Simple split for words
            if words:  # Check if words list is not empty
                all_words.extend([word for word in words[1:] if word != 'the'])

        if not all_words:
            print(f"No valid words in column {column_name} of {csv_file}. Skipping...")
            continue

        # Count word frequencies
        word_freq = Counter(all_words)

        # Create frequency dictionary for word cloud
        word_freq_dict = dict(word_freq)

        # Generate word cloud
        wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate_from_frequencies(word_freq_dict)
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.title(f'Word Cloud of {column_name} ({dataset_name})')
        plt.axis('off')
        wordcloud_output_path = os.path.join(output_folder, f'{dataset_name}_{column_name.lower()}_word_cloud.png')
        plt.savefig(wordcloud_output_path, bbox_inches='tight', dpi=300)
        plt.close()
        print(f"Saved word cloud for {column_name} ({dataset_name}) to '{wordcloud_output_path}'")

Saved word cloud for Question (train) to './data-overview-analysis/new/world_cloud/train_question_word_cloud.png'
Saved word cloud for Answer (train) to './data-overview-analysis/new/world_cloud/train_answer_word_cloud.png'
Saved word cloud for Question (validation) to './data-overview-analysis/new/world_cloud/validation_question_word_cloud.png'
Saved word cloud for Answer (validation) to './data-overview-analysis/new/world_cloud/validation_answer_word_cloud.png'
Saved word cloud for Question (test) to './data-overview-analysis/new/world_cloud/test_question_word_cloud.png'
Saved word cloud for Answer (test) to './data-overview-analysis/new/world_cloud/test_answer_word_cloud.png'


# Pie Chart - Visualization Type of Question

In [18]:
output_folder = './data-overview-analysis/new/pie_chart/'
os.makedirs(output_folder, exist_ok=True)

In [24]:
# Column to analyze
column_name = 'Question'

for csv_file in csv_files:
    # Read CSV
    file_path = os.path.join(input_folder, csv_file)
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"File {file_path} not found. Skipping...")
        continue

    dataset_name = csv_file.replace('.csv', '')

    if column_name not in df.columns:
        print(f"Column {column_name} not found in {csv_file}. Skipping...")
        continue

    # Split questions if multiple per row (assuming comma-separated)
    all_questions = []
    for question_str in df[column_name].astype(str):
        questions = question_str.split(', ')
        all_questions.extend(questions)

    if not all_questions:
        print(f"No valid questions in column {column_name} of {csv_file}. Skipping...")
        continue

    # Get question type based on first word
    def get_question_type(question):
        tokens = word_tokenize(question.strip())
        return tokens[0].lower() if tokens else "unknown"

    question_types = [get_question_type(q) for q in all_questions]
    type_counts = Counter(question_types)

    # Group types with <5% into "Others"
    total_questions = sum(type_counts.values())
    threshold = 0.05  # 5%
    significant_types = {}
    others_count = 0
    for q_type, count in type_counts.items():
        percentage = count / total_questions
        if percentage >= threshold:
            significant_types[q_type] = count
        else:
            others_count += count

    if others_count > 0:
        significant_types['Others'] = others_count

    # Create pie chart
    labels = list(significant_types.keys())
    sizes = list(significant_types.values())
    plt.figure(figsize=(8, 8))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=plt.cm.Pastel1.colors)
    plt.title(f'Question Type Distribution - {dataset_name} ')
    output_path = os.path.join(output_folder, f'{dataset_name}_question_types.png')
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"Saved pie chart for {column_name} ({dataset_name}) to '{output_path}'")

Saved pie chart for Question (train) to './data-overview-analysis/new/pie_chart/train_question_types.png'
Saved pie chart for Question (validation) to './data-overview-analysis/new/pie_chart/validation_question_types.png'
Saved pie chart for Question (test) to './data-overview-analysis/new/pie_chart/test_question_types.png'
